# Gruppenarbeit Star Type Classification Dataset

Gruppe: Mona Auer, Leni Biasi, Dana Kando

Ziel dieses Notebooks ist es, das Star Type Classification Dataset zu laden und zu explorieren, Datenprobleme zu dokumentieren, eine reproduzierbare Preprocessing-Pipeline zu entwickeln und Machine-Learning-Modelle fuer die Vorhersage der Sternklasse `Type` zu trainieren und zu evaluieren.

## 1. Problemstellung

Das Dataset enthält physikalische und spektrale Eigenschaften von Sternen. Die zentrale Machine-Learning-Aufgabe ist eine **multiklassige Klassifikation**: Aus den Merkmalen `Temperature`, `L`, `R`, `A_M`, `Color` und `Spectral_Class` soll die Zielvariable `Type` vorhergesagt werden.

Die Klassen sind:

| Type | Bedeutung |
| --- | --- |
| 0 | Red Dwarf |
| 1 | Brown Dwarf |
| 2 | White Dwarf |
| 3 | Main Sequence |
| 4 | Super Giants |
| 5 | Hyper Giants |

In [ ]:
# Wir laden hier alle wichtigen Bibliotheken, die wir im Notebook brauchen.
from pathlib import Path

# NumPy brauchen wir für bestimmte Werte wie np.nan.
import numpy as np

# Pandas brauchen wir, um mit Tabellen und CSV-Dateien zu arbeiten.
import pandas as pd

# Matplotlib verwenden wir für Diagramme.
import matplotlib.pyplot as plt

# Seaborn verwenden wir fuer uebersichtlichere Diagramme.
import seaborn as sns

# Diese Funktionen und Modelle brauchen wir spaeter fuer die Machine-Learning-Pipeline.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

# Wir stellen hier den allgemeinen Stil für unsere Diagramme ein.
sns.set_theme(style="whitegrid", context="notebook")

# Wir setzen einen festen Zufallswert, damit unsere Ergebnisse reproduzierbar bleiben.
RANDOM_STATE = 42

## 2. Dataset laden

Die Datei ist nicht als Standard-CSV gespeichert. Das erkennt man an mehreren Punkten:

- Das Trennzeichen ist `~`, nicht `,`.
- Die erste Zeile ist eine Beschreibung und nicht der Header.
- Rechts neben den relevanten Spalten befinden sich leere Zusatzspalten.
- Durch leere Zeilen entstehen beim Laden zunaechst zusaetzliche Datensaetze ohne Zielvariable.

Deshalb wird die Datei gezielt geladen: `sep="~"`, `skiprows=1` und `usecols=range(7)`, damit nur die sieben relevanten Spalten verwendet werden.

In [ ]:
# Wir speichern hier den Pfad zu unserer CSV-Datei.
# Die Datei liegt praktischerweise im gleichen Ordner wie unser Notebook.
DATA_PATH = Path("nasa_stars_data.csv")

# Wir lesen die CSV-Datei mit pandas ein.
# Die Datei ist mit "~" getrennt, deshalb geben wir sep="~" an.
# Mit skiprows=1 überspringen wir die erste Beschreibungszeile.
# Mit usecols=range(7) nehmen wir nur die ersten sieben wichtigen Spalten.
raw = pd.read_csv(DATA_PATH, sep="~", skiprows=1, header=0, usecols=range(7), engine="python")

# Wir geben den Spalten eigene, einheitliche Namen.
raw.columns = ["Temperature", "L", "R", "A_M", "Color", "Spectral_Class", "Type"]

# Wir schauen uns die ersten fuenf Zeilen an, um zu pruefen, ob das Einlesen funktioniert hat.
raw.head()

In [ ]:
# Wir schauen uns mal an, wie viele Zeilen und Spalten unser Datensatz hat.
raw.shape

## 3. Erste Datenpruefung

Wir pruefen Datentypen, fehlende Werte, Zielklassenverteilung und auffaellige Kategorien.

In [ ]:
# Wir lassen uns allgemeine Informationen zum Datensatz anzeigen.
# So sehen wir zum Beispiel Datentypen und fehlende Werte.
raw.info()

In [ ]:
# Wir zählen jetzt, wie viele fehlende Werte es in jeder Spalte gibt.
raw.isna().sum()

In [ ]:
# Wir schauen, wie oft jede Sternklasse in der Zielvariable Type vorkommt.
# sort_index() sortiert die Klassen nach ihrer Nummer.
raw["Type"].value_counts().sort_index()

In [ ]:
# Wir schauen uns an, welche Farben im Datensatz vorkommen.
# dropna=False sorgt dafuer, dass auch fehlende Werte mitgezaehlt werden.
raw["Color"].value_counts(dropna=False)

In [ ]:
# Wir schauen uns an, welche Spektralklassen im Datensatz vorkommen.
# Auch hier zaehlen wir fehlende Werte mit.
raw["Spectral_Class"].value_counts(dropna=False)

In [ ]:
# Wir legen fest, welche Spalten numerische Werte enthalten.
numeric_cols = ["Temperature", "L", "R", "A_M"]

# Wir lassen uns statistische Kennzahlen für diese numerischen Spalten anzeigen.
# Das .T dreht die Tabelle, damit sie übersichtlicher ist, hat uns Chat Gpt erklärt :)
raw[numeric_cols].describe().T

### Befunde aus der Exploration

- Beim Laden entstehen 275 Zeilen, davon sind 35 komplett leere Zusatzzeilen ohne Zielvariable. Fuer das Machine Learning bleiben 240 gueltige Beobachtungen uebrig.
- Die Zielvariable ist bei den gueltigen Beobachtungen perfekt balanciert: jede Klasse kommt 40-mal vor.
- Es gibt fehlende Werte in `Temperature`, `R`, `Color` und `Spectral_Class`.
- `Color` ist uneinheitlich kodiert, z. B. `Blue-white`, `Blue White`, `Blue white`, `Blue-White`, `Blue-wh!te`, `Blu`, `white` und ` White`.
- Die numerischen Variablen `L` und `R` besitzen sehr unterschiedliche Groessenordnungen und starke Spannweiten. Skalierung ist daher besonders fuer lineare Modelle und SVM wichtig.
- Die Originaldatei enthaelt technische Ladeprobleme: falsches Standard-Trennzeichen, vorgeschaltete Textzeile und leere Zusatzspalten.

## 4. Visualisierung

Die folgenden Diagramme helfen uns, Unterschiede zwischen den Sternklassen sichtbar zu machen.

In [ ]:
# Wir und Chat Gpt erstellen eine Grafik mit 2 Zeilen und 2 Spalten für unsere vier numerischen Merkmale.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Wir machen aus den Achsen eine einfache Liste, damit wir leichter darüber iterieren können.
axes = axes.ravel()

# Wir erstellen für jedes numerische Merkmal einen Boxplot.
for ax, col in zip(axes, numeric_cols):
    # Der Boxplot zeigt, wie sich die Werte je nach Sternklasse unterscheiden.
    sns.boxplot(data=raw, x="Type", y=col, ax=ax)

    # Wir geben jedem Diagramm einen passenden Titel.
    ax.set_title(f"{col} nach Sternklasse")

# Wir sorgen dafür, dass sich die Diagramme nicht überlappen.
plt.tight_layout()

In [ ]:
# Wir legen die Größe der Grafik fest.
plt.figure(figsize=(8, 5))

# Wir erstellen ein Streudiagramm mit Temperatur und absoluter Magnitude.
# Die Farben zeigen dabei die verschiedenen Sternklassen.
sns.scatterplot(data=raw, x="Temperature", y="A_M", hue="Type", palette="tab10")

# Wir geben dem Diagramm einen Titel.
plt.title("Temperatur und absolute Magnitude nach Sternklasse")

# Wir sorgen dafür, dass alles sauber dargestellt wird.
plt.tight_layout()

## 5. Cleaning-Strategie

Die Originaldatei wird nicht manuell ueberschrieben. Stattdessen wird eine reproduzierbare Cleaning-Funktion definiert:

- Komplett leere Zeilen und Zeilen ohne Zielvariable `Type` werden entfernt.
- Leere Strings werden als fehlende Werte behandelt.
- Numerische Spalten werden sicher in Zahlen konvertiert.
- `Color` wird vereinheitlicht, z. B. werden Schreibfehler und verschiedene Schreibweisen zusammengefuehrt.
- `Spectral_Class` wird vereinheitlicht und fehlende Werte werden spaeter im Pipeline-Schritt imputiert.
- Fehlende numerische Werte werden mit dem Median imputiert.
- Kategoriale Werte werden mit dem haeufigsten Wert imputiert und per One-Hot-Encoding codiert.

In [ ]:
# Wir schreiben eine eigene Funktion, um den Datensatz zu bereinigen.
def clean_star_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    # Wir erstellen zuerst eine Kopie, damit wir die Originaldaten nicht verändern.
    cleaned = df.copy()

    # Wir gehen jede Spalte im Datensatz durch.
    for col in cleaned.columns:
        # Wenn die Spalte Text enthält, bereinigen wir die Textwerte.
        if cleaned[col].dtype == "object":
            # Wir wandeln die Werte in Text um und entfernen Leerzeichen am Anfang und Ende.
            cleaned[col] = cleaned[col].astype("string").str.strip()

            # Leere Texte ersetzen wir durch fehlende Werte.
            cleaned[col] = cleaned[col].replace({"": pd.NA})

    # Wir wandeln die numerischen Spalten wirklich in Zahlen um.
    for col in ["Temperature", "L", "R", "A_M", "Type"]:
        # Wenn ein Wert nicht in eine Zahl umgewandelt werden kann, wird daraus ein fehlender Wert.
        cleaned[col] = pd.to_numeric(cleaned[col], errors="coerce")

    # Wir erstellen eine Zuordnung, um unterschiedliche Schreibweisen der Farben zu vereinheitlichen.
    color_map = {
        "blu": "Blue",
        "blue": "Blue",
        "blue white": "Blue-white",
        "blue-white": "Blue-white",
        "blue-wh!te": "Blue-white",
        "white": "White",
        "whitish": "White",
        "white-yellow": "Yellow-white",
        "yellowish white": "Yellow-white",
        "yellow-white": "Yellow-white",
        "yellowish": "Yellowish",
        "pale yellow orange": "Pale yellow orange",
        "orange": "Orange",
        "orange-red": "Orange-red",
        "red": "Red",
    }

    # Wir machen die Farbnamen zuerst klein und ersetzen Unterstriche durch Leerzeichen.
    normalized_color = cleaned["Color"].str.lower().str.replace("_", " ", regex=False).str.replace(r"\s+", " ", regex=True)

    # Wir ersetzen die unterschiedlichen Farbschreibweisen durch einheitliche Namen.
    cleaned["Color"] = normalized_color.map(color_map).fillna(cleaned["Color"])

    # Wir schreiben die Spektralklassen einheitlich in Großbuchstaben.
    cleaned["Spectral_Class"] = cleaned["Spectral_Class"].str.upper()

    # Wir sorgen dafür, dass fehlende Werte bei den Textspalten korrekt als np.nan gespeichert werden.
    cleaned[["Color", "Spectral_Class"]] = cleaned[["Color", "Spectral_Class"]].astype(object).where(
        cleaned[["Color", "Spectral_Class"]].notna(), np.nan
    )


    # Komplett leere Zeilen und Zeilen ohne Zielvariable koennen nicht fuer ueberwachtes Lernen verwendet werden.
    cleaned = cleaned.dropna(how="all")
    cleaned = cleaned.dropna(subset=["Type"])

    # Wir wandeln die Zielvariable Type in ganze Zahlen um.
    cleaned["Type"] = cleaned["Type"].astype("Int64")

    # Wir geben den bereinigten Datensatz zurück.
    return cleaned

# Wir wenden unsere Bereinigungsfunktion auf die Rohdaten an.
data = clean_star_dataframe(raw)

# Wir schauen uns jetzt die ersten fünf Zeilen des bereinigten Datensatzes an .
data.head()

In [ ]:
# Wir prüfen nach der Bereinigung noch einmal, wo fehlende Werte vorhanden sind.
data.isna().sum()


In [ ]:
# Wir schauen uns die bereinigten Farbkategorien an.
# Dadurch sehen wir, ob die Vereinheitlichung funktioniert hat.
data["Color"].value_counts(dropna=False)

## 6. Machine-Learning-Pipeline

Wir verwenden eine `ColumnTransformer`-Pipeline. Das verhindert Data Leakage, weil Imputation, Skalierung und One-Hot-Encoding nur auf den Trainingsdaten gelernt und anschliessend auf Validierungs- bzw. Testdaten angewendet werden.

In [ ]:
# Wir trennen die Eingabedaten von der Zielvariable.
X = data.drop(columns="Type")

# Unsere Zielvariable ist Type, also die Sternklasse.
y = data["Type"].astype(int)

# Wir legen fest, welche Merkmale numerisch sind.
numeric_features = ["Temperature", "L", "R", "A_M"]

# Wir legen fest, welche Merkmale kategorial sind.
categorical_features = ["Color", "Spectral_Class"]

# Wir erstellen eine Pipeline fuer die numerischen Spalten.
numeric_preprocess = Pipeline(steps=[
    # Fehlende numerische Werte ersetzen wir durch den Median.
    ("imputer", SimpleImputer(strategy="median")),

    # Danach skalieren wir die numerischen Werte.
    ("scaler", StandardScaler()),
])

# Wir erstellen eine Pipeline fuer die kategorialen Spalten.
categorical_preprocess = Pipeline(steps=[
    # Fehlende Textwerte ersetzen wir durch den haeufigsten Wert.
    ("imputer", SimpleImputer(strategy="most_frequent")),

    # Danach wandeln wir Kategorien in Zahlen-Spalten um.
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

# Wir verbinden die numerische und kategoriale Vorverarbeitung.
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_preprocess, numeric_features),
    ("cat", categorical_preprocess, categorical_features),
])

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Support Vector Machine": SVC(kernel="rbf", C=10, gamma="scale", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = []
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=["accuracy", "f1_macro"], return_train_score=False)
    results.append({
        "model": name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
        "f1_macro_mean": scores["test_f1_macro"].mean(),
        "f1_macro_std": scores["test_f1_macro"].std(),
    })

results_df = pd.DataFrame(results).sort_values("f1_macro_mean", ascending=False)
results_df

## 7. Finales Modell evaluieren

Nach dem Modellvergleich trainieren wir das beste Modell auf einem Trainingssplit und bewerten es auf einem separaten Testsplit. Wegen der kleinen Datenmenge ist Cross Validation wichtiger als ein einzelner Split; der Split hilft aber, eine konkrete Confusion Matrix und einen Classification Report zu zeigen.

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

final_pipeline = Pipeline(steps=[("preprocess", preprocessor), ("model", best_model)])
final_pipeline.fit(X_train, y_train)
y_pred = final_pipeline.predict(X_test)

print(f"Bestes Modell laut Cross Validation: {best_model_name}")
print(f"Accuracy Testset: {accuracy_score(y_test, y_pred):.3f}")
print(f"Macro F1 Testset: {f1_score(y_test, y_pred, average='macro'):.3f}")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=sorted(y.unique()), yticklabels=sorted(y.unique()))
plt.xlabel("Vorhergesagte Klasse")
plt.ylabel("Tatsaechliche Klasse")
plt.title(f"Confusion Matrix: {best_model_name}")
plt.tight_layout()

## 8. Interpretation und Fazit

Die Sternklassifikation eignet sich gut als ueberwachtes Lernproblem, weil die Zielvariable `Type` fuer die gueltigen Beobachtungen vorhanden und balanciert ist. Die wichtigsten Vorverarbeitungsschritte sind das robuste Laden der Datei, das Entfernen leerer Zusatzzeilen, die Vereinheitlichung von Farbkategorien, die Behandlung fehlender Werte und die Skalierung numerischer Features.

Fuer die Bewertung werden **Accuracy** und **Macro F1** verwendet. Accuracy ist wegen der balancierten Klassen gut interpretierbar. Macro F1 ist trotzdem sinnvoll, weil jede Klasse gleich stark gewichtet wird und damit schwache Leistung in einzelnen Klassen sichtbar bleibt.

Aufgrund der kleinen Datenmenge von 240 Beobachtungen sollte die Modellqualitaet nicht nur ueber einen einzelnen Train-Test-Split beurteilt werden. Die 5-fache stratified Cross Validation liefert eine stabilere Schaetzung.

## 9. Grenzen und moegliche Verbesserungen

- Das Dataset ist klein, wodurch Ergebnisse je nach Split schwanken koennen.
- Einige fehlende Werte koennten astrophysikalisch fundierter imputiert werden, z. B. anhand aehnlicher Sternklassen oder physikalischer Beziehungen.
- `Color` enthaelt Schreibfehler und uneinheitliche Labels. Die hier genutzte Mapping-Tabelle ist transparent, aber fachlich diskutierbar.
- Eine Hyperparameter-Optimierung mit `GridSearchCV` oder `RandomizedSearchCV` koennte die Modelle weiter verbessern.
- Feature Engineering mit logarithmierten Werten fuer `L` und `R` koennte sinnvoll sein, weil diese Variablen sehr grosse Wertebereiche besitzen.

## 10. Dokumentation der LLM-Nutzung

Fuer dieses Projekt wurde ein Large Language Model als Unterstuetzung verwendet. Genutzt wurde es fuer:

- Strukturierung des Notebooks und des Abschlussberichts
- Vorschlaege fuer eine geeignete Machine-Learning-Strategie
- Formulierung einer reproduzierbaren Cleaning- und Preprocessing-Pipeline
- Hinweise zur Dokumentation typischer Datenprobleme

Die Verantwortung fuer Code, Interpretation und finale Abgabe liegt vollstaendig bei der Gruppe. Alle Gruppenmitglieder muessen die Schritte im Notebook nachvollziehen und erklaeren koennen.